## 📥 Twitter Sentiment Analysis
In this lab, you will build a tweet classification system using Natural Language Processing (NLP) techniques to classify Twitter tweets as either positive or negative.

🔍 What you will do:
- Load the dataset
- Split the data and vectorize the tweets
- Build and train a classification model
- Evaluate the model’s performance
- Test the model with new tweet examples

### 🛠️ Import library

In [20]:
!pip install datasets scikit-learn textblob -q
!python -m textblob.download_corpora kagglehub
!pip install kagglehub

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
Finished.


In [21]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from textblob import TextBlob

import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.porter import PorterStemmer

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

### 🔤📄 Data Loading

In [22]:
df = pd.read_csv('https://raw.githubusercontent.com/iiFadel/majal-ai-track-2026/main/labs/day4/data/tweet_sentiment.csv')

df.head()

,tweet,sentiment
0,The event starts at 5 PM.,neutral
1,I hate how this turned out.,negative
2,Fantastic experience!,positive
3,Fantastic experience!,positive
4,This is the worst thing ever!,negative


### 🧮 Data Splitting & Vectorization

In [23]:
# 🧽 Define a function to clean the tweet text
english_stopwords = stopwords.words('english')
stemmer = PorterStemmer()

def clean_review(text):
  # Tokenize the text and filter out non-alphabetic words
  words = [word for word in word_tokenize(text) if word.isalpha()]

  # Remove stopwords and stem the remaining words
  cleaned_words = [stemmer.stem(word) for word in words if word not in english_stopwords]

  # Join the cleaned words into a single string
  text = ' '.join(cleaned_words)

  return text

In [24]:
# 🧹 Apply cleaning function to tweetsdf['cleaned_tweet'] = df['text'].apply(clean_review)

df["cleaned_tweet"] = df["tweet"].apply(clean_review)

# Display original and cleaned tweets
df[["tweet", "cleaned_tweet", "sentiment"]].head()



,tweet,cleaned_tweet,sentiment
0,The event starts at 5 PM.,the event start pm,neutral
1,I hate how this turned out.,i hate turn,negative
2,Fantastic experience!,fantast experi,positive
3,Fantastic experience!,fantast experi,positive
4,This is the worst thing ever!,thi worst thing ever,negative


In [25]:
# 🎯 Prepare data for training

X = df["cleaned_tweet"]
y = df["sentiment"]

# ✂️ Split data into training and testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 🔢 Vectorize the text data

vectorizer = CountVectorizer()

X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print("Training shape:", X_train_vectorized.shape)
print("Testing shape:", X_test_vectorized.shape)



Training shape: (800, 45)
Testing shape: (200, 45)


### 🌐🤖 Model Building & Evaluation

In [26]:
# 🤖 Train a logistic regression classifier

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_vectorized, y_train)

# 📊 Evaluate training performance

train_predictions = model.predict(X_train_vectorized)

train_accuracy = accuracy_score(
    y_train,
    train_predictions
)

print("Training Accuracy:", train_accuracy)


Training Accuracy: 1.0


In [27]:
# 📊 Evaluate test performance


test_predictions = model.predict(X_test_vectorized)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

print("Testing Accuracy:", test_accuracy)


Testing Accuracy: 1.0


### 🔍🎭 Testing

In [28]:
# ✏️ Correct spelling errors in custom tweets

def correct_spelling(text):
    corrected_text = TextBlob(text).correct()
    return str(corrected_text)


# 🔍 Define function to make sentiment prediction on custom review

def predict_sentiment(tweet):
    # Correct spelling
    corrected_tweet = correct_spelling(tweet)

    # Clean the tweet
    cleaned_tweet = clean_review(corrected_tweet)

    # Convert the tweet into numbers
    tweet_vectorized = vectorizer.transform([cleaned_tweet])

    # Predict the sentiment
    prediction = model.predict(tweet_vectorized)[0]

    print("Original tweet:", tweet)
    print("Corrected tweet:", corrected_tweet)
    print("Cleaned tweet:", cleaned_tweet)
    print("Predicted sentiment:", prediction)

    return prediction


In [29]:
# 🧪 Test the model with custom tweets

predict_sentiment("I hte this terrible movie")


Original tweet: I hte this terrible movie
Corrected tweet: I the this terrible movie
Cleaned tweet: i terribl movi
Predicted sentiment: negative


'negative'